In [1]:
import sys
sys.path.append(r'C:\Users\ian32\Downloads\final\main.py')  # 加入你的 main.py 路徑

from main import RTBBiddingSystem  # 假設你的主類別叫這個

In [2]:
system = RTBBiddingSystem(student_id='your_id')

In [3]:
system.load_data()

載入資料中...
訓練集大小: (1760309, 22)
測試集大小: (436648, 20)


In [4]:
system.prepare_training_data()

準備訓練資料...
選定的特徵數量: 41
訓練樣本數: 1760309
click 標籤分布：
click
0    0.999251
1    0.000749
Name: proportion, dtype: float64


In [5]:
system.train_ctr_model()

訓練 CTR 模型...
類別不平衡分析：
- 負樣本 (未點擊): 1758990 (99.93%)
- 正樣本 (已點擊): 1319 (0.07%)
- 不平衡比率: 1333.6:1
執行資料採樣，平衡正負樣本比例...
採樣後訓練集大小: 2110, 正樣本比例: 50.00%
開始模型訓練...
[LightGBM] [Info] Number of positive: 1055, number of negative: 1055
[LightGBM] [Debug] Dataset::GetMultiBinFromSparseFeatures: sparse rate 0.860496
[LightGBM] [Debug] Dataset::GetMultiBinFromAllFeatures: sparse rate 0.464758
[LightGBM] [Debug] init for col-wise cost 0.001218 seconds, init for row-wise cost 0.002000 seconds
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002508 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 795
[LightGBM] [Info] Number of data points in the train set: 2110, number of used features: 38
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Debug] Trained a tree with leaves = 30 and depth = 8
Training unti

In [6]:
system.train_winprice_model()

訓練 Win-Price 模型...
警告：Win-Price 模型的欄位 'anonymous_url_id' 中存在 NaN 值。
正在用中位數/0 (0) 填充 'anonymous_url_id' 中的 NaN。
訓練最終 Win-Price 模型...
⚠️ 生成模型報告時發生錯誤: unsupported format string passed to Series.__format__


In [14]:
import pandas as pd

# 假設 system.train_data 已經存在且有 timestamp 欄位

# 1. 先檢查 hour 欄位是否已存在
if 'hour' not in system.train_data.columns:
    if 'timestamp' in system.train_data.columns:
        # 2. 嘗試最快速的自動格式推斷
        try:
            system.train_data['hour'] = pd.to_datetime(
                system.train_data['timestamp'],
                errors='coerce',
                infer_datetime_format=True
            ).dt.hour
        except Exception as e:
            print(f"自動推斷失敗，嘗試 format='mixed'，錯誤訊息：{e}")
            try:
                # pandas 2.0+ 支援 format='mixed'
                system.train_data['hour'] = pd.to_datetime(
                    system.train_data['timestamp'],
                    errors='coerce',
                    format='mixed'
                ).dt.hour
            except Exception as e2:
                print(f"format='mixed' 也失敗，錯誤訊息：{e2}")
                # 3. 最後嘗試只取前19碼（去掉毫秒與時區）
                system.train_data['timestamp_clean'] = system.train_data['timestamp'].astype(str).str.slice(0, 19)
                system.train_data['hour'] = pd.to_datetime(
                    system.train_data['timestamp_clean'],
                    errors='coerce'
                ).dt.hour
    else:
        raise ValueError("train_data 缺少 hour 及 timestamp 欄位，無法分時段統計")

# 檢查結果
print(system.train_data[['timestamp', 'hour']].head())
print("hour 欄位 NaN 數量：", system.train_data['hour'].isna().sum())

                          timestamp  hour
0  2013-06-06 00:01:04.828000+00:00   0.0
1  2013-06-06 00:01:05.075000+00:00   0.0
2  2013-06-06 00:01:05.119000+00:00   0.0
3  2013-06-06 00:01:05.254000+00:00   0.0
4  2013-06-06 00:01:05.284000+00:00   0.0
hour 欄位 NaN 數量： 1759


In [15]:
system.bid_day2()

執行 Day2 出價...
預處理整個測試集進行出價...
測試集預處理完成。
計算歷史績效並優化預算分配...
已處理 50000/436648 筆競價請求. 剩餘總預算: 4780.00. 出價次數: 52. 假設花費: 220.00
已處理 100000/436648 筆競價請求. 剩餘總預算: 4780.00. 出價次數: 52. 假設花費: 220.00
已處理 150000/436648 筆競價請求. 剩餘總預算: 4780.00. 出價次數: 52. 假設花費: 220.00
已處理 200000/436648 筆競價請求. 剩餘總預算: 4780.00. 出價次數: 52. 假設花費: 220.00
已處理 250000/436648 筆競價請求. 剩餘總預算: 4780.00. 出價次數: 52. 假設花費: 220.00
已處理 300000/436648 筆競價請求. 剩餘總預算: 4780.00. 出價次數: 52. 假設花費: 220.00


KeyboardInterrupt: 

In [ ]:
system.generate_budget_allocation_report()